# Étape 5 — TTA + Ensemble + Calibration (Phase A)

**Constat étape 4** :

| Modèle | val score | err_F | err_M | gap | gap / score |
|---|---|---|---|---|---|
| baseline mbv3s | 0.00122 | 0.00064 | 0.00103 | 0.00039 | 32 % |
| F1 effb0 | 0.00120 | 0.00072 | 0.00104 | 0.00032 | 27 % |
| **F2 convnext_tiny** | **0.00116** | 0.00043 | 0.00092 | 0.00049 | **42 %** |

## 1. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()   # racine du depot (les notebooks sont a la racine)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader

from dc_utils.paths import IMAGE_DIR, ARTIFACTS_DIR
from dc_utils.training import OcclusionDataset, default_eval_tf
from dc_utils.metric import compute_score, weighted_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device, '| timm', timm.__version__, '| torch', torch.__version__)

EXP_DIR = ROOT / 'experiments'
df_val = pd.read_csv(ROOT / 'data' / 'val_split.csv')
print('val:', len(df_val), '| n_F:', (df_val.gender == 0.0).sum(), '| n_M:', (df_val.gender == 1.0).sum())


<repo>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device = cuda | timm 1.0.27 | torch 2.2.2+cu121
val: 9977 | n_F: 3229 | n_M: 6748


## 2. Reload F1 (efficientnet_b0) et F2 (convnext_tiny)

In [2]:
# Même TimmRegressor que l'étape 4. On charge des poids déjà entraînés,
# donc bias_init est inutile ici (mais on garde l'API cohérente).
class TimmRegressor(nn.Module):
    def __init__(self, backbone_name: str, pretrained: bool = False, bias_init: float | None = None):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=1)
        if bias_init is not None:
            head = self.backbone.get_classifier()
            with torch.no_grad():
                head.bias.fill_(bias_init); head.weight.mul_(0.01)
    def forward(self, x):
        return torch.sigmoid(self.backbone(x)).squeeze(-1)

def load_ckpt(backbone_name: str, ckpt_path: Path) -> nn.Module:
    m = TimmRegressor(backbone_name, pretrained=False).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(state['model'])
    m.eval()
    print(f'loaded {ckpt_path.name} | backbone={backbone_name} | best ep={state.get("epoch")} | val={state.get("val")}')
    return m

model_F1 = load_ckpt('efficientnet_b0', ARTIFACTS_DIR / 'exp_F1_effb0.pt')
model_F2 = load_ckpt('convnext_tiny',   ARTIFACTS_DIR / 'exp_F2_convnext.pt')


loaded exp_F1_effb0.pt | backbone=efficientnet_b0 | best ep=7 | val={'score': 0.001062619900285066, 'err_F': 0.0005125854015615943, 'err_M': 0.000879275067377242, 'gap': 0.00036668966581564775, 'n_F': 3229, 'n_M': 6748}
loaded exp_F2_convnext.pt | backbone=convnext_tiny | best ep=7 | val={'score': 0.0011637669447581373, 'err_F': 0.0004344179381005615, 'err_M': 0.0009206506092056121, 'gap': 0.0004862326711050506, 'n_F': 3229, 'n_M': 6748}


## 3. Helper de prédiction

Fonction qui, étant donné un modèle et un DataLoader, retourne un `np.ndarray` de prédictions dans le **même ordre** que `df_val`. Pour le TTA flip on calcule $\frac{1}{2}(\text{pred}(x) + \text{pred}(\text{flip}(x)))$.

`OcclusionDataset(has_target=True)` retourne `(x, y, gender, filename)`. On utilise `batch[0]` (input) et `batch[-1]` (filenames).

In [3]:
@torch.inference_mode()
def predict(model, dl, device, tta_flip: bool = False, use_amp: bool = True):
    model.eval()
    preds, fns = [], []
    ctx = torch.amp.autocast('cuda', enabled=use_amp)
    for batch in dl:
        X = batch[0].to(device, non_blocking=True)
        fn = batch[-1]
        with ctx:
            p = model(X).float()
            if tta_flip:
                p_flip = model(torch.flip(X, dims=[-1])).float()
                p = 0.5 * (p + p_flip)
        preds.append(p.cpu().numpy()); fns.extend(fn)
    return np.concatenate(preds), fns

ds_val = OcclusionDataset(df_val, IMAGE_DIR, default_eval_tf(), has_target=True)
dl_val = DataLoader(ds_val, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

def df_with_pred(df_base: pd.DataFrame, preds: np.ndarray, fns: list[str]) -> pd.DataFrame:
    out = pd.DataFrame({'filename': fns, 'pred': preds})
    out = (df_base[['filename','FaceOcclusion','gender']]
           .rename(columns={'FaceOcclusion':'target'})
           .merge(out, on='filename'))
    assert len(out) == len(df_base), f'merge size mismatch: {len(out)} vs {len(df_base)}'
    return out


## 4. F1, F2, avec/sans TTA

In [4]:
import time

RECIPES: dict[str, pd.DataFrame] = {}

def run(name: str, model, tta_flip: bool):
    t0 = time.time()
    p, fns = predict(model, dl_val, device, tta_flip=tta_flip)
    df_pred = df_with_pred(df_val, p, fns)
    s = compute_score(df_pred)
    dt = time.time() - t0
    print(f'{name:25s} | score {s["score"]:.5f} | err_F {s["err_F"]:.5f} | err_M {s["err_M"]:.5f} | gap {s["gap"]:.5f} | {dt:.0f}s')
    RECIPES[name] = df_pred
    return s

_ = run('F1_effb0',          model_F1, tta_flip=False)
_ = run('F2_convnext',       model_F2, tta_flip=False)
_ = run('F1_effb0_TTA',      model_F1, tta_flip=True)
_ = run('F2_convnext_TTA',   model_F2, tta_flip=True)


F1_effb0                  | score 0.00120 | err_F 0.00072 | err_M 0.00104 | gap 0.00032 | 18s
F2_convnext               | score 0.00116 | err_F 0.00043 | err_M 0.00092 | gap 0.00049 | 15s
F1_effb0_TTA              | score 0.00119 | err_F 0.00070 | err_M 0.00102 | gap 0.00033 | 13s
F2_convnext_TTA           | score 0.00113 | err_F 0.00043 | err_M 0.00089 | gap 0.00047 | 15s


## 5. Ensemble F1 + F2 (moyenne pondérée)

In [5]:
def ensemble_preds(*dfs: pd.DataFrame, weights: list[float] | None = None) -> pd.DataFrame:
    base = dfs[0][['filename','target','gender']].copy()
    P = np.stack([d.set_index('filename').loc[base['filename'], 'pred'].values for d in dfs])
    if weights is None:
        weights = [1.0/len(dfs)] * len(dfs)
    base['pred'] = (np.array(weights)[:,None] * P).sum(axis=0)
    return base

def show(name, df_pred):
    s = compute_score(df_pred)
    print(f'{name:32s} | score {s["score"]:.5f} | err_F {s["err_F"]:.5f} | err_M {s["err_M"]:.5f} | gap {s["gap"]:.5f}')
    RECIPES[name] = df_pred
    return s

_ = show('ENS_F1+F2',           ensemble_preds(RECIPES['F1_effb0'],     RECIPES['F2_convnext']))
_ = show('ENS_F1+F2_TTA',       ensemble_preds(RECIPES['F1_effb0_TTA'], RECIPES['F2_convnext_TTA']))
# Variantes pondérées : convnext est meilleur en val seul, on lui donne plus de poids.
_ = show('ENS_0.3F1+0.7F2_TTA', ensemble_preds(RECIPES['F1_effb0_TTA'], RECIPES['F2_convnext_TTA'], weights=[0.3, 0.7]))
_ = show('ENS_0.4F1+0.6F2_TTA', ensemble_preds(RECIPES['F1_effb0_TTA'], RECIPES['F2_convnext_TTA'], weights=[0.4, 0.6]))


ENS_F1+F2                        | score 0.00104 | err_F 0.00046 | err_M 0.00085 | gap 0.00038
ENS_F1+F2_TTA                    | score 0.00103 | err_F 0.00046 | err_M 0.00084 | gap 0.00038
ENS_0.3F1+0.7F2_TTA              | score 0.00104 | err_F 0.00042 | err_M 0.00083 | gap 0.00041
ENS_0.4F1+0.6F2_TTA              | score 0.00103 | err_F 0.00043 | err_M 0.00083 | gap 0.00040


## 6. Calibration isotonique par genre — 5-fold CV

In [6]:
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold

def kfold_isotonic_per_gender(df_pred: pd.DataFrame, n_splits: int = 5, seed: int = 42) -> pd.DataFrame:
    df = df_pred.reset_index(drop=True).copy()
    df['pred_cal'] = np.nan
    for g_value in [0.0, 1.0]:
        idx_g = df.index[df['gender'] == g_value].to_numpy()
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_local, te_local in kf.split(idx_g):
            tr = idx_g[tr_local]; te = idx_g[te_local]
            iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
            iso.fit(df.loc[tr,'pred'].values, df.loc[tr,'target'].values)
            df.loc[te, 'pred_cal'] = iso.predict(df.loc[te, 'pred'].values)
    out = df[['filename','target','gender']].copy()
    out['pred'] = df['pred_cal'].values
    return out

def calib(name_in: str, name_out: str):
    df_cal = kfold_isotonic_per_gender(RECIPES[name_in])
    return show(name_out, df_cal)

print('--- Calibration isotonique par genre (5-fold honnête) ---')
_ = calib('F2_convnext_TTA',      'F2_TTA + iso(g)')
_ = calib('F1_effb0_TTA',         'F1_TTA + iso(g)')
_ = calib('ENS_F1+F2_TTA',        'ENS_TTA + iso(g)')
_ = calib('ENS_0.3F1+0.7F2_TTA',  'ENS_0.3/0.7_TTA + iso(g)')
_ = calib('ENS_0.4F1+0.6F2_TTA',  'ENS_0.4/0.6_TTA + iso(g)')


--- Calibration isotonique par genre (5-fold honnête) ---
F2_TTA + iso(g)                  | score 0.00119 | err_F 0.00044 | err_M 0.00094 | gap 0.00050
F1_TTA + iso(g)                  | score 0.00127 | err_F 0.00073 | err_M 0.00109 | gap 0.00036
ENS_TTA + iso(g)                 | score 0.00110 | err_F 0.00048 | err_M 0.00089 | gap 0.00041
ENS_0.3/0.7_TTA + iso(g)         | score 0.00109 | err_F 0.00044 | err_M 0.00088 | gap 0.00043
ENS_0.4/0.6_TTA + iso(g)         | score 0.00110 | err_F 0.00046 | err_M 0.00089 | gap 0.00043


## 7. calibration isotonique **globale** (sans split par genre)

On veut vérifier si la calibration isotonique avec le split par genre est utile

In [7]:
def kfold_isotonic_global(df_pred: pd.DataFrame, n_splits: int = 5, seed: int = 42) -> pd.DataFrame:
    df = df_pred.reset_index(drop=True).copy()
    df['pred_cal'] = np.nan
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, te in kf.split(df):
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        iso.fit(df.loc[tr,'pred'].values, df.loc[tr,'target'].values)
        df.loc[te, 'pred_cal'] = iso.predict(df.loc[te, 'pred'].values)
    out = df[['filename','target','gender']].copy()
    out['pred'] = df['pred_cal'].values
    return out

print('--- Calibration isotonique globale (5-fold) ---')
for name_in in ['F2_convnext_TTA', 'ENS_F1+F2_TTA', 'ENS_0.3F1+0.7F2_TTA']:
    df_cal = kfold_isotonic_global(RECIPES[name_in])
    show(f'{name_in} + iso(global)', df_cal)


--- Calibration isotonique globale (5-fold) ---
F2_convnext_TTA + iso(global)    | score 0.00117 | err_F 0.00043 | err_M 0.00092 | gap 0.00049
ENS_F1+F2_TTA + iso(global)      | score 0.00107 | err_F 0.00049 | err_M 0.00088 | gap 0.00039
ENS_0.3F1+0.7F2_TTA + iso(global) | score 0.00108 | err_F 0.00044 | err_M 0.00087 | gap 0.00043


## 8. Tableau de synthèse

In [8]:
rows = []
for name, df_pred in RECIPES.items():
    s = compute_score(df_pred)
    rows.append({'recipe': name, **s})
df_recipes = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
ref_score = df_recipes.loc[df_recipes['recipe']=='F2_convnext','score'].iloc[0]
df_recipes['gain_vs_F2'] = df_recipes['score'] - ref_score
df_recipes['gain_%']     = (-df_recipes['gain_vs_F2'] / ref_score * 100).round(2)
df_recipes['gap_%']      = (df_recipes['gap'] / df_recipes['score'] * 100).round(1)
df_recipes.to_csv(EXP_DIR / 'step5_phaseA_recipes.csv', index=False)
df_recipes


,recipe,score,err_F,err_M,gap,n_F,n_M,gain_vs_F2,gain_%,gap_%
0,ENS_0.4F1+0.6F2_TTA,0.001029,0.000435,0.000831,0.000396,3229,6748,-0.000135,11.59,38.5
1,ENS_F1+F2_TTA,0.001030,0.000458,0.000839,0.000381,3229,6748,-0.000134,11.53,37.0
2,ENS_F1+F2,0.001038,0.000463,0.000846,0.000383,3229,6748,-0.000126,10.79,36.9
3,ENS_0.3F1+0.7F2_TTA,0.001038,0.000420,0.000832,0.000412,3229,6748,-0.000125,10.77,39.7
4,ENS_F1+F2_TTA + iso(global),0.001070,0.000488,0.000876,0.000388,3229,6748,-0.000094,8.06,36.2
5,ENS_0.3F1+0.7F2_TTA + iso(global),0.001080,0.000440,0.000867,0.000427,3229,6748,-0.000083,7.17,39.5
6,ENS_0.3/0.7_TTA + iso(g),0.001092,0.000444,0.000876,0.000432,3229,6748,-0.000071,6.14,39.5
7,ENS_TTA + iso(g),0.001099,0.000482,0.000893,0.000411,3229,6748,-0.000065,5.58,37.4
8,ENS_0.4/0.6_TTA + iso(g),0.001102,0.000456,0.000886,0.000431,3229,6748,-0.000062,5.32,39.1
9,F2_convnext_TTA,0.001129,0.000426,0.000895,0.000469,3229,6748,-0.000034,2.96,41.5


## 9. Sauvegarde du pipeline d'inférence

In [ ]:
import pickle, re

# Choix automatique de la meilleure recette 
BEST_RECIPE = df_recipes.iloc[0]['recipe']
print(f'Best recipe (val) : {BEST_RECIPE}  -> score {df_recipes.iloc[0]["score"]:.5f}')

USE_CALIB = ' + iso' in BEST_RECIPE
USE_TTA   = '_TTA' in BEST_RECIPE
USE_ENS   = BEST_RECIPE.startswith('ENS')
print(f'USE_TTA={USE_TTA} | USE_ENS={USE_ENS} | USE_CALIB={USE_CALIB}')

def fit_iso_per_gender_full(df_pred: pd.DataFrame) -> dict:
    iso_by_g = {}
    for g_value in [0.0, 1.0]:
        m = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        sub = df_pred[df_pred['gender'] == g_value]
        m.fit(sub['pred'].values, sub['target'].values)
        iso_by_g[g_value] = m
    return iso_by_g

def fit_iso_global_full(df_pred: pd.DataFrame) -> IsotonicRegression:
    m = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
    m.fit(df_pred['pred'].values, df_pred['target'].values)
    return m

def apply_iso_per_gender(df_pred: pd.DataFrame, iso_by_g: dict) -> pd.DataFrame:
    out = df_pred.copy()
    for g_value, m in iso_by_g.items():
        mask = out['gender'] == g_value
        out.loc[mask, 'pred'] = m.predict(out.loc[mask, 'pred'].values)
    return out

# Recette de base (sans le suffixe ' + iso(*)')
base_recipe = re.sub(r' \+ iso.*$', '', BEST_RECIPE)
df_uncal = RECIPES[base_recipe]

iso_per_gender = None
iso_global     = None
if USE_CALIB:
    if 'iso(global)' in BEST_RECIPE:
        iso_global = fit_iso_global_full(df_uncal)
        df_final_val = df_uncal.copy()
        df_final_val['pred'] = iso_global.predict(df_final_val['pred'].values)
    else:
        iso_per_gender = fit_iso_per_gender_full(df_uncal)
        df_final_val = apply_iso_per_gender(df_uncal, iso_per_gender)
else:
    df_final_val = df_uncal

print('Pipeline final (refit iso sur 100% val) :', compute_score(df_final_val))

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)
pipeline_path = ARTIFACTS_DIR / 'pipeline_phaseA.pkl'
with open(pipeline_path, 'wb') as f:
    pickle.dump({
        'best_recipe': BEST_RECIPE,
        'use_tta': USE_TTA,
        'use_ensemble': USE_ENS,
        'use_calib': USE_CALIB,
        'iso_per_gender': iso_per_gender,
        'iso_global': iso_global,
        'val_score_final': compute_score(df_final_val),
        'ckpts': {
            'F1_effb0':    str(ARTIFACTS_DIR / 'exp_F1_effb0.pt'),
            'F2_convnext': str(ARTIFACTS_DIR / 'exp_F2_convnext.pt'),
        },
    }, f)
print(f'pipeline saved -> {pipeline_path}')
df_final_val.to_csv(EXP_DIR / 'preds_val_pipelineA.csv', index=False)


Best recipe (val) : ENS_0.4F1+0.6F2_TTA  -> score 0.00103
USE_TTA=True | USE_ENS=True | USE_CALIB=False
Pipeline final (refit iso sur 100% val) : {'score': 0.0010288120861806427, 'err_F': 0.0004347722312332308, 'err_M': 0.0008307988011981721, 'gap': 0.00039602656996494135, 'n_F': 3229, 'n_M': 6748}
pipeline saved -> <repo>/artifacts/pipeline_phaseA.pkl
